<div style="background:linear-gradient(120deg,#0B1F3A 0%,#123A4C 55%,#00B4A6 140%);color:#F6F1E7;padding:28px 32px;border-radius:18px;">
<p style="letter-spacing:.18em;text-transform:uppercase;font-size:12px;opacity:.8;margin:0;">Playground Series · S6E9 · EV Purchase Lab</p>
<h1 style="margin:10px 0 8px;font-size:34px;line-height:1.12;">The 0.58% switch behind 0.946</h1>
<p style="margin:0;font-size:17px;opacity:.92;max-width:42rem;">No subsidy almost forbids a Yes. Twenty-five recorded experiments later, the honest model sits at <b>0.94628</b>. The public <b>0.94649</b> is a file stack, not a new DGP.</p>
<p style="margin:14px 0 0;font-size:13px;opacity:.75;">Den Pugovkin · journal tables with Great Tables · numbers cited from a frozen 5-fold log</p>
</div>


<div style="background:#F6F1E7;border:1px solid #D6CDB8;border-radius:16px;padding:20px 24px;margin:16px 0;">
<p style="letter-spacing:.14em;text-transform:uppercase;font-size:11px;color:#00B4A6;margin:0 0 10px;">How to read this notebook</p>
<p style="margin:0 0 12px;line-height:1.55;color:#1C1917;">This is <a href="https://www.kaggle.com/competitions/playground-series-s6e9" style="color:#0B1F3A;"><b>Playground Series S6E9</b></a>: predict <code>P(Will_Buy_EV = Yes)</code>. The metric is <b>ROC AUC</b> (higher is better). It is not a classification accuracy contest.</p>
<p style="margin:0 0 12px;line-height:1.55;color:#1C1917;"><b>Why this title.</b> <i>0.58%</i> is the purchase rate when <code>Subsidy_Available = No</code> — almost a kill switch, not a mild effect. <i>0.946</i> is the AUC people chase on the public leaderboard. The claim is that the switch is the mechanism sitting under that number, not “yet another LightGBM.”</p>
<p style="margin:0 0 12px;line-height:1.55;color:#1C1917;"><b>What was done.</b> A private lab ran 25 controlled experiments on the same frozen Stratified 5-fold split (seed 42). Honest trained result: a 3-seed LightGBM bag at <b>CV 0.946061 / public 0.94628</b>. Public <b>0.94649</b> is a rank mix of already-scored CSVs — a file, not a new data-generating process. Mixing extra original rows or weaker rankings <b>hurt</b> or tied.</p>
<p style="margin:0;line-height:1.55;color:#1C1917;"><b>What this kernel produces.</b> It computes the switchboard live, trains <b>one</b> leak-free LightGBM (digits + fold-local target rates on the same knobs), and writes <code>submission.csv</code>. If nina’s high-band prediction files are attached, the official file is their rank stack — the lab already showed that mixing a weaker trained ranking into 0.94649 <b>loses</b> public. The trained model is always saved as <code>submission_trained.csv</code>.</p>
</div>


<div style="background:#F6F1E7;border-left:6px solid #00B4A6;padding:16px 20px;border-radius:0 14px 14px 0;margin:14px 0;">
<b>TL;DR</b>
<ol style="margin:8px 0 0;padding-left:1.15rem;line-height:1.55;">
<li><code>Subsidy_Available=No</code> → <b>0.58%</b> Yes. <code>Range_Anxiety_Level=High</code> → <b>0.14%</b> Yes. Gender is a decoy.</li>
<li>A published synthetic recipe already scores <b>~0.9377 AUC</b> as one number, with no trees.</li>
<li>Digits + nested target rates + LightGBM close most of the gap: honest bag <b>CV 0.946061 / public 0.94628</b> (<code>exp019</code>).</li>
<li>Public <b>0.94649</b> is a rank mix of scored CSVs (<code>exp018</code>). Dosing it with a weaker ranking loses the fourth decimal.</li>
<li>This kernel trains one LightGBM and writes <code>submission.csv</code>. High-band community files, if attached, are stacked — not mixed into the bag.</li>
</ol>
</div>

Internet On for <code>great_tables</code>. Attach Playground S6E9. Optional: <code>nina2025/ps-s6e9-11</code> and <code>nina2025/ps-s6e9-13</code> (only <code>0.94649.csv</code>, <code>0.94644.csv</code>, <code>0.94639.csv</code>). Do not attach the original 10k table as extra train — that run lost.


## Contents

1. [Contract](#1) — rows, metric, no missingness
2. [The 0.58% switch](#2) — subsidy × anxiety, live from train
3. [A recipe already scores 0.938](#3) — published constants, not fitted here
4. [Twenty-five experiments, one table](#4) — what actually moved
5. [Honest 0.94628 vs file 0.94649](#5) — what to keep, what to stop
6. [Train the switchboard model](#6) — leak-free LightGBM, writes a file
7. [High-band stack](#7) — community CSVs only if they are stronger


<h2 id="1" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">1. Contract</h2>

Question: is this a clean probability table, or a messy join?


In [ ]:
%pip install -q great_tables


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from great_tables import GT, loc, style
from IPython.display import HTML, display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

INK, PAPER, NAVY = "#1C1917", "#F6F1E7", "#0B1F3A"
TEAL, GOLD, CORAL = "#00B4A6", "#F4C95D", "#E85D4C"
PALETTE = [TEAL, NAVY, GOLD, CORAL, "#5B8DEF"]
HEAT = [CORAL, PAPER, TEAL]
LAYOUT = dict(
    paper_bgcolor=PAPER,
    plot_bgcolor=PAPER,
    font=dict(color=INK, size=13),
    margin=dict(l=48, r=24, t=56, b=40),
    height=480,
)


def draw(fig):
    fig.update_layout(height=480)
    display(fig)


In [ ]:
def journal(frame, title, subtitle, note="", rowname=None, groupname=None):
    tbl = GT(frame, rowname_col=rowname, groupname_col=groupname)
    tbl = (
        tbl.tab_header(title=title, subtitle=subtitle)
        .tab_style(style.fill(color=NAVY), loc.header())
        .tab_style(style.text(color=PAPER), loc.header())
        .tab_style(style.fill(color="#123A4C"), loc.column_labels())
        .tab_style(style.text(color=PAPER), loc.column_labels())
        .tab_style(style.fill(color=TEAL), loc.row_groups())
        .tab_style(style.text(color=PAPER, weight="bold"), loc.row_groups())
        .tab_options(
            table_background_color=PAPER,
            table_font_color=INK,
            table_font_names=["Georgia", "Times New Roman", "serif"],
            table_border_top_color=TEAL,
            table_border_top_width="4px",
            heading_background_color=NAVY,
            heading_align="left",
            column_labels_background_color="#123A4C",
            row_group_background_color=TEAL,
            source_notes_background_color=PAPER,
            row_striping_include_table_body=True,
            row_striping_background_color="#EFE8D8",
            table_width="100%",
            data_row_padding="7px",
        )
    )
    return tbl.tab_source_note(note) if note else tbl


def show(tbl):
    display(HTML(tbl.as_raw_html()))


In [ ]:
SLUG = "playground-series-s6e9"
roots = [
    Path(f"/kaggle/input/competitions/{SLUG}"),
    Path(f"/kaggle/input/{SLUG}"),
    Path("competitions/predicting-electric-vehicle-purchases/data/raw"),
    Path("data/raw"),
]
DATA = next((p for p in roots if (p / "train.csv").is_file()), None)
if DATA is None:
    raise FileNotFoundError("Attach Playground Series S6E9 (train.csv).")
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
y = (train["Will_Buy_EV"].astype(str) == "Yes").astype(int)
print("data", DATA)
print("train", train.shape, "test", test.shape)


In [ ]:
snap = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(train), len(test)],
        "yes_rate": [float(y.mean()), np.nan],
        "missing": [int(train.isna().sum().sum()), int(test.isna().sum().sum())],
        "id_min": [int(train.id.min()), int(test.id.min())],
        "id_max": [int(train.id.max()), int(test.id.max())],
    }
)
show(
    journal(
        snap,
        "S6E9 is a clean probability table",
        "ROC AUC on P(Will_Buy_EV = Yes). Higher is better.",
        "Sample submission is the train rate on every test id — AUC 0.5, not a model.",
        rowname="split",
    )
    .cols_label(
        rows="Rows",
        yes_rate="P(Yes)",
        missing="Missing cells",
        id_min="id min",
        id_max="id max",
    )
    .fmt_integer(["rows", "missing", "id_min", "id_max"])
    .fmt_percent("yes_rate", decimals=2)
    .sub_missing(missing_text="—")
)


Decision: no grouping key, no dates, no missing values, no train/test ID overlap. Stratified 5-fold on the label is the honest split. Accuracy is the wrong metric here.


<h2 id="2" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">2. The 0.58% switch</h2>

Question: which categoricals almost turn the label off?


In [ ]:
def rates(col: str) -> pd.DataFrame:
    return (
        train.assign(yes=y)
        .groupby(col, observed=False)
        .agg(n=("yes", "size"), yes_rate=("yes", "mean"))
        .reset_index()
        .rename(columns={col: "level"})
        .assign(feature=col)[["feature", "level", "n", "yes_rate"]]
        .sort_values("yes_rate")
    )


switches = pd.concat(
    [
        rates("Gender"),
        rates("Subsidy_Available"),
        rates("Range_Anxiety_Level"),
    ],
    ignore_index=True,
)
show(
    journal(
        switches,
        "Two kill switches. One decoy.",
        "Same label, three categoricals. Only two of them matter.",
        f"Live on {len(train):,} train rows in this kernel.",
        rowname="level",
        groupname="feature",
    )
    .cols_label(n="Rows", yes_rate="P(Yes)")
    .fmt_integer("n")
    .fmt_percent("yes_rate", decimals=2)
    .data_color("yes_rate", palette=HEAT)
)


In [ ]:
joint = (
    train.assign(yes=y)
    .groupby(["Subsidy_Available", "Range_Anxiety_Level"], observed=False)
    .agg(n=("yes", "size"), yes_rate=("yes", "mean"))
    .reset_index()
    .sort_values("yes_rate")
)
show(
    journal(
        joint,
        "The DGP is a 2 × 3 switchboard",
        "Joint purchase rates, not two independent main effects.",
        "This is why single-column target encoding was not the last honest lift.",
    )
    .cols_label(
        Subsidy_Available="Subsidy",
        Range_Anxiety_Level="Range anxiety",
        n="Rows",
        yes_rate="P(Yes)",
    )
    .fmt_integer("n")
    .fmt_percent("yes_rate", decimals=2)
    .data_color("yes_rate", palette=HEAT)
)


In [ ]:
grid = joint.pivot(
    index="Range_Anxiety_Level",
    columns="Subsidy_Available",
    values="yes_rate",
).loc[["High", "Medium", "Low"]]
fig = px.imshow(
    grid,
    color_continuous_scale=HEAT,
    title="Joint P(Yes): subsidy × range anxiety",
    text_auto=".2%",
)
fig.update_layout(**LAYOUT, coloraxis_colorbar=dict(title="P(Yes)", tickformat=".0%"))
draw(fig)


Decision: a model that ignores subsidy or range anxiety is leaving the data-generating process on the table. The interesting remainder is the **joint** cells, not another chart of Gender.


<h2 id="3" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">3. A recipe already scores 0.938</h2>

Question: how far do the reported synthetic knobs go **without trees**?

Coefficients from Chris Deotte’s original-table write-up. Constants. Not fitted on these folds.


In [ ]:
income = train["Annual_Income_USD"].astype(float)
concern = train["Environmental_Concern_Level"].astype(float)
subsidy_yes = (train["Subsidy_Available"].astype(str) == "Yes").astype(float)
medium = (train["Range_Anxiety_Level"].astype(str) == "Medium").astype(float)
high = (train["Range_Anxiety_Level"].astype(str) == "High").astype(float)
recipe = (
    1.2 * (income / 100_000.0)
    + 0.6 * concern
    + 2.0 * subsidy_yes
    - 1.0 * medium
    - 3.0 * high
)
recipe_auc = float(roc_auc_score(y, recipe))
knobs = (
    train.assign(yes=y)
    .groupby("Will_Buy_EV")[["Annual_Income_USD", "Environmental_Concern_Level"]]
    .mean()
    .reset_index()
)


In [ ]:
show(
    journal(
        knobs,
        "Income and concern move with the label",
        f"Recipe-only AUC in this kernel: {recipe_auc:.6f}",
        "score = 1.2·(income/1e5) + 0.6·concern + 2·subsidy − 1·medium − 3·high",
        rowname="Will_Buy_EV",
    )
    .cols_label(
        Annual_Income_USD="Mean income (USD)",
        Environmental_Concern_Level="Mean concern",
    )
    .fmt_number("Annual_Income_USD", decimals=0, use_seps=True)
    .fmt_number("Environmental_Concern_Level", decimals=2)
)
fig = px.histogram(
    pd.DataFrame({"recipe": recipe, "label": np.where(y == 1, "Yes", "No")}),
    x="recipe",
    color="label",
    barmode="overlay",
    nbins=64,
    color_discrete_map={"No": NAVY, "Yes": TEAL},
    title="The recipe already ranks buyers (~0.938 AUC).",
    opacity=0.72,
)
fig.update_layout(**LAYOUT)
draw(fig)


Decision: trees still beat the recipe (~0.9418 vs 0.9377). Shipping `recipe_score` as an extra XGB column did **not** help (<code>exp004</code>, CV 0.941807, public 0.94155). The leftover is representation, not that linear score.


<h2 id="4" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">4. Twenty-five experiments, one table</h2>

Question: which steps actually moved ROC AUC on frozen v1?

CV numbers below are **cited** from the lab log (Stratified 5-fold, seed 42, fingerprint <code>f50fdbcd…</code>). This kernel does not retrain them. Public LB is ~20% of test — fourth decimals can twitch.


In [ ]:
ladder = pd.DataFrame(
    [
        ("Foundation", "exp001 HGB", 0.941451, 0.94132, "keep"),
        ("Foundation", "exp003 XGB raw", 0.941814, 0.94159, "keep"),
        ("Representation", "exp008 digits+freq", 0.943908, 0.94400, "keep"),
        ("Representation", "exp011 nested TE", 0.945782, 0.94600, "keep"),
        ("Representation", "exp014 LightGBM", 0.945997, 0.94616, "keep"),
        ("Representation", "exp015 extra TE", 0.946039, 0.94625, "keep"),
        ("Honest model", "exp019 3-seed bag", 0.946061, 0.94628, "keep"),
        ("File stack", "exp018 high-band", np.nan, 0.94649, "keep public file"),
        ("Closed door", "exp016 CatBoost", 0.945893, 0.94595, "reject"),
        ("Closed door", "exp024 OOF mix", 0.946100, 0.94625, "reject"),
        ("Closed door", "exp025 original 10k", 0.946027, 0.94621, "reject"),
    ],
    columns=["band", "experiment", "cv", "public", "decision"],
)
show(
    journal(
        ladder,
        "What moved — and what only looked like it moved",
        "Frozen v1 · ROC AUC maximize · one material change per row",
        "Cited from the competition log. File stacks have no OOF. Do not retune to 0.94649.",
        rowname="experiment",
        groupname="band",
    )
    .cols_label(cv="CV", public="Public", decision="Decision")
    .tab_spanner(label="ROC AUC", columns=["cv", "public"])
    .fmt_number(["cv", "public"], decimals=5)
    .sub_missing(missing_text="—")
    .data_color("public", palette=HEAT)
)


In [ ]:
fig = px.scatter(
    ladder.dropna(subset=["cv"]),
    x="cv",
    y="public",
    color="band",
    text="experiment",
    color_discrete_sequence=PALETTE,
    title="Public tracks honest CV — until the file stack.",
)
fig.update_traces(textposition="top center", marker=dict(size=11))
fig.update_layout(**LAYOUT, xaxis_title="CV AUC", yaxis_title="Public AUC")
draw(fig)


Decision: the large honest lifts are digits, nested income/commute rates, and LightGBM. Extra singleton TE keys added **+0.000042 CV**. Neighborhood, CatBoost, a residual MLP, community OOF, and the original 10k table did not replace the bag.


<h2 id="5" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">5. Honest 0.94628 vs file 0.94649</h2>

Question: which file should stay selected, and which ideas are closed?


In [ ]:
keep = pd.DataFrame(
    [
        ("(28).csv", "exp018 rank H-blend", np.nan, 0.94649, "public file"),
        ("trained.csv", "exp019 3-seed LightGBM", 0.946061, 0.94628, "honest hedge"),
        ("(25).csv", "exp015 single-seed LGBM", 0.946039, 0.94625, "parent"),
        ("(35).csv", "exp025 original extra-train", 0.946027, 0.94621, "do not select"),
        ("(34).csv", "exp024 80/20 community OOF", 0.946100, 0.94625, "do not select"),
    ],
    columns=["file", "what_it_is", "cv", "public", "role"],
)
show(
    journal(
        keep,
        "Keep (28).csv selected until something beats 0.94649",
        "The hedge is a trained model. The leader is a ranked mix of scored CSVs.",
        "Checkbox unverified in the log. You choose the final two.",
        rowname="file",
    )
    .cols_label(what_it_is="What it is", cv="CV", public="Public", role="Role")
    .tab_spanner(label="ROC AUC", columns=["cv", "public"])
    .fmt_number(["cv", "public"], decimals=5)
    .sub_missing(missing_text="—")
    .data_color("public", palette=HEAT)
)


In [ ]:
closed = pd.DataFrame(
    {
        "idea": [
            "recipe_score as an XGB column",
            "2,000 raw XGB rounds",
            "CatBoost on the 74-feature matrix",
            "mix the 0.94628 bag into 0.94649",
            "residual MLP",
            "community OOF onto the bag",
            "original 10k extra-fold",
        ],
        "result": [
            "exp004 public 0.94155",
            "exp005 public 0.94161",
            "exp016 public 0.94595, 0/5",
            "18% → 0.94647; 10% → 0.94648",
            "exp021 public 0.94549, 0/5",
            "exp024 public 0.94625",
            "exp025 public 0.94621, 0/5",
        ],
    }
)
show(
    journal(
        closed,
        "Closed doors — do not rerun under a new name",
        "Negative results are part of the score.",
        "Original extra-train shifted frequencies the wrong way. Mixing weaker rankings dilutes 0.94649.",
        rowname="idea",
    )
    .cols_label(result="What happened")
)


<h2 id="6" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">6. Train the switchboard model</h2>

Question: what is the strongest **honest** file this kernel can learn from train?

One LightGBM. Frozen Stratified 5-fold, seed 42. Encoders fit on fold-train only. Digits + smoothed target rates on income/commute and the two switches. This is the exp014/exp015 family, not the 3-seed bag and not a public-file mix. CPU, expect ~15–25 minutes.


In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

SEED = 42
CATS = [
    "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level",
]
NUMS = [
    "Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
LGBM = dict(
    n_estimators=3500, learning_rate=0.02, max_depth=5, num_leaves=32,
    min_child_samples=10, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0, max_bin=255,
    random_state=SEED, n_jobs=4, verbose=-1, metric="auc",
)


In [ ]:
def digits(frame: pd.DataFrame) -> pd.DataFrame:
    income = np.rint(frame["Annual_Income_USD"].to_numpy(float)).astype(np.int64)
    tenths = np.rint(frame["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)
    commute = frame["Daily_Commute_km"].to_numpy(float)
    out = pd.DataFrame(index=frame.index)
    for p in (1, 10, 100, 1000):
        out[f"income_digit_{p}"] = (income // p) % 10
    for m in (10, 100, 1000):
        out[f"income_mod_{m}"] = income % m
        out[f"income_round_{m}"] = (income % m == 0).astype(np.int8)
    out["commute_tenth"] = tenths % 10
    out["commute_integer"] = np.floor(commute)
    out["commute_fraction"] = commute - np.floor(commute)
    return out


def te_keys(frame: pd.DataFrame) -> pd.DataFrame:
    income = frame["Annual_Income_USD"].astype(float)
    commute = frame["Daily_Commute_km"].astype(float)
    keys = pd.DataFrame(
        {
            "income_exact": income.astype(str),
            "income_floor100": np.floor(income / 100).astype(str),
            "income_floor1000": np.floor(income / 1000).astype(str),
            "commute_exact": commute.astype(str),
            "commute_integer": np.floor(commute).astype(str),
            "switch": (
                frame["Subsidy_Available"].astype(str)
                + "|"
                + frame["Range_Anxiety_Level"].astype(str)
            ),
        },
        index=frame.index,
    )
    for col in CATS:
        keys[col] = frame[col].astype(str)
    return keys


In [ ]:
def nested_rate(keys: pd.Series, y_col: pd.Series, m: float = 10.0) -> np.ndarray:
    out = np.zeros(len(keys), dtype=np.float32)
    splitter = KFold(5, shuffle=True, random_state=SEED)
    for fit_i, hold_i in splitter.split(keys):
        prior = float(y_col.iloc[fit_i].mean())
        stats = (
            pd.DataFrame({"k": keys.iloc[fit_i], "y": y_col.iloc[fit_i]})
            .groupby("k")["y"].agg(["sum", "count"])
        )
        mapped = (stats["sum"] + m * prior) / (stats["count"] + m)
        out[hold_i] = keys.iloc[hold_i].map(mapped).fillna(prior).to_numpy()
    return out


def oos_rate(tr_keys: pd.Series, y_col: pd.Series, out_keys: pd.Series, m=10.0):
    prior = float(y_col.mean())
    stats = pd.DataFrame({"k": tr_keys, "y": y_col}).groupby("k")["y"].agg(["sum", "count"])
    mapped = (stats["sum"] + m * prior) / (stats["count"] + m)
    return out_keys.map(mapped).fillna(prior).astype(np.float32)


In [ ]:
def fold_xy(raw_tr, y_tr, raw_out):
    keys_tr, keys_out = te_keys(raw_tr), te_keys(raw_out)
    x = raw_out[NUMS + CATS].copy()
    for col in CATS:
        levels = sorted(raw_tr[col].astype(str).unique())
        x[col] = pd.Categorical(raw_out[col].astype(str), categories=levels)
    x = pd.concat([x, digits(raw_out)], axis=1)
    same = raw_out is raw_tr
    for col in keys_tr.columns:
        for m in (10.0, 100.0):
            tag = f"te_{col}_s{m:g}"
            if same:
                x[tag] = nested_rate(keys_tr[col], y_tr, m)
            else:
                x[tag] = oos_rate(keys_tr[col], y_tr, keys_out[col], m)
    return x


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(train), dtype=float)
test_sum = np.zeros(len(test), dtype=float)
fold_rows = []
for fold, (tr, va) in enumerate(folds.split(train, y)):
    raw_tr, raw_va = train.iloc[tr], train.iloc[va]
    y_tr, y_va = y.iloc[tr], y.iloc[va]
    X_tr = fold_xy(raw_tr, y_tr, raw_tr)
    X_va = fold_xy(raw_tr, y_tr, raw_va)
    X_te = fold_xy(raw_tr, y_tr, test)
    xtr, xes, ytr, yes = train_test_split(
        X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED
    )
    model = LGBMClassifier(**LGBM)
    model.fit(
        xtr, ytr, eval_set=[(xes, yes)],
        callbacks=[early_stopping(100), log_evaluation(200)],
    )
    oof[va] = model.predict_proba(X_va)[:, 1]
    test_sum += model.predict_proba(X_te)[:, 1]
    auc = float(roc_auc_score(y_va, oof[va]))
    trees = int(getattr(model, "best_iteration_", 0) or 0)
    fold_rows.append({"fold": fold, "auc": auc, "trees": trees})
    print(f"fold {fold}  {auc:.6f}  trees {trees}", flush=True)
cv = float(roc_auc_score(y, oof))
trained = test_sum / 5
print("OOF", f"{cv:.6f}")


In [ ]:
fold_tbl = pd.DataFrame(fold_rows)
show(
    journal(
        fold_tbl,
        "Leak-free LightGBM on the switchboard",
        f"OOF AUC {cv:.6f} · lab bag 0.946061 is the 3-seed ceiling, not this cell",
        "Encoders never see fold-valid targets. Official lab number remains exp019 if versions drift.",
        rowname="fold",
    )
    .cols_label(auc="Fold AUC", trees="Trees")
    .fmt_number("auc", decimals=6)
    .fmt_integer("trees")
    .data_color("auc", palette=HEAT)
)
trained_path = Path("/kaggle/working/submission_trained.csv")
if not trained_path.parent.exists():
    trained_path = Path("submission_trained.csv")
trained_sub = sample.copy()
trained_sub["Will_Buy_EV"] = trained
trained_sub.to_csv(trained_path, index=False)
print("wrote", trained_path)


<h2 id="7" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">7. High-band stack</h2>

Question: how do other people’s files raise public **without** poisoning the trained ranking?

The lab already answered this. Rank-mix **only** the high band (0.94649 / 0.94644 / 0.94639). Do not mix the bag, RealMLP, original 10k rows, or 0.94614-class clones into that stack — every measured dose lost a fourth decimal. Attach <code>nina2025/ps-s6e9-11</code> and <code>nina2025/ps-s6e9-13</code>. If they are missing, the official file is the trained model.


In [ ]:
import hashlib


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_csv(name: str):
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    hits = [p for p in root.rglob(name) if p.is_file()]
    return hits[0] if hits else None


BAND = [
    ("0.94649.csv", 0.58, "381e5c9d11e6e0cdc706a96cd0971f9b47bf4411f4d9923835927c477e5060ca"),
    ("0.94644.csv", 0.30, "277a6c3e680d82fb633b27901ce2e77d3d5a382587e094bb3864cafffa1658bc"),
    ("0.94639.csv", 0.12, "a3e40fcde1e555ef109c71d88ca55859802a563e1d277158b27f51c2181b4911"),
]


In [ ]:
inventory, found = [], []
for name, weight, pin in BAND:
    path = find_csv(name)
    row = {"file": name, "weight": weight, "found": path is not None, "hash_ok": False}
    if path is not None and file_sha256(path) == pin:
        frame = pd.read_csv(path)
        assert list(frame.columns) == ["id", "Will_Buy_EV"]
        assert frame["id"].equals(sample["id"])
        row["hash_ok"] = True
        found.append((weight, frame))
    inventory.append(row)
show(
    journal(
        pd.DataFrame(inventory),
        "Community files as scored rankings — not extra train rows",
        "Pins are the lab hashes for nina 0.94649 / 0.94644 and Naji 0.94639.",
        "Original 10k EV table is not an input. exp025 already rejected extra-train.",
        rowname="file",
    )
    .cols_label(weight="Blend weight", found="Found", hash_ok="Hash matches")
)


In [ ]:
def rank_map(frames, weights, reference):
    ranks = [w * f["Will_Buy_EV"].rank(method="average") for w, f in zip(weights, frames)]
    mixed = sum(ranks[1:], ranks[0])
    order = np.argsort(np.argsort(mixed.to_numpy()))
    return np.sort(reference["Will_Buy_EV"].to_numpy())[order]


out = Path("/kaggle/working/submission.csv")
if not out.parent.exists():
    out = Path("submission.csv")
if len(found) == 3:
    weights, frames = zip(*found)
    official = rank_map(frames, weights, frames[0])
    source = "high-band rank stack mapped onto 0.94649"
elif len(found) == 1:
    official = found[0][1]["Will_Buy_EV"].to_numpy()
    source = "single pinned high-band file"
else:
    official = trained
    source = "trained LightGBM (high-band files missing or unpinned)"
sub = sample.copy()
sub["Will_Buy_EV"] = official
assert sub["id"].equals(sample["id"])
assert np.isfinite(sub["Will_Buy_EV"]).all()
sub.to_csv(out, index=False)
print("official", out, source, "mean", float(sub["Will_Buy_EV"].mean()))


Decision: the research output is two files. `submission_trained.csv` is the model this notebook learned. `submission.csv` is the public file — high-band stack when the pinned CSVs are present, otherwise the trained model. Do not blend those two together.


<div style="background:#0B1F3A;color:#F6F1E7;padding:22px 26px;border-radius:16px;margin-top:12px;">
<p style="letter-spacing:.14em;text-transform:uppercase;font-size:11px;opacity:.7;margin:0;">Reproduce</p>
<p style="margin:8px 0 0;">Competition: <a href="https://www.kaggle.com/competitions/playground-series-s6e9" style="color:#00B4A6;">playground-series-s6e9</a></p>
<p style="margin:6px 0 0;">Cited experiments: <code>exp001</code>–<code>exp025</code>, frozen v1, seed 42.</p>
<p style="margin:6px 0 0;">Live: switch rates, recipe AUC, one LightGBM, optional high-band stack.</p>
<p style="margin:6px 0 0;">Inputs: competition data + optional <code>nina2025/ps-s6e9-11</code> and <code>nina2025/ps-s6e9-13</code>.</p>
<p style="margin:6px 0 0;">Not used: original 10k extra-train, mixing the bag into 0.94649.</p>
<p style="margin:14px 0 0;opacity:.75;font-size:13px;">Great Tables · Plotly · LightGBM. No vote-begging.</p>
</div>
